## HIL-005 Data Cleaning

This notebook documents the cleaning and validation process for dataset HIL-005.  The raw source data will remain unchanged, and all cleaned outputs will be saved separately.

Source: City of Hillsboro GIS

Dataset: Buildings (<1,000)

Layer ID: 91

Geometry: Polygon

Spatial Reference: WKID 3857

Source documentation: [\[gis.hillsboro-oregon.gov\]](https://gis.hillsboro-oregon.gov/public/rest/services/public/Planning_BaseData/MapServer/91)

## Findings from the HIL_005_Buildings_Exploration Notebook

- The layer contains 43,686 building records.
- Geometry type is polygon.
- The current dataset contains no Demoed or Permitted records.
- `YEAR_BUILT = 0` occurs in 8,732 records (~20% of the dataset) and should be treated as a potential missing/unknown value rather than a literal construction year.
- `YEAR_DEMOLISHED` is not populated in the current dataset.
- The `STATUS` field uses a coded domain: 0 = Active, 1 = Demoed, 2 = Permitted.
- The current dataset therefore appears most useful for analyzing the existing building stock rather than historical demolition activity.

In [1]:
from pathlib import Path

# Establish the project root
PROJECT_ROOT = Path.cwd().parent

# Locate available dated data folders
DATA_FOLDERS = sorted(
    [
        folder
        for folder in PROJECT_ROOT.iterdir()
        if folder.is_dir() and folder.name.startswith("20")
    ]
)

print("Available data folders:")
for folder in DATA_FOLDERS:
    print("-", folder.name)

Available data folders:
- 2026-08-26


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Change to the desired data version here
DATA_VERSION = "2026-08-26"

DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"

print("Using data version:", DATA_VERSION)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Using data version: 2026-08-26
Raw data directory: c:\Users\John\Documents\hillsborogis\2026-08-26\raw
Processed data directory: c:\Users\John\Documents\hillsborogis\2026-08-26\processed


In [3]:
# List files in the raw data directory
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))

HIL-001.json
HIL-002.json
HIL-003.json
HIL-004.json
HIL-005.json
HIL-006.json
HIL-007.json
HIL-008.json
HIL-009.json


In [4]:
import json

# Define the raw HIL-005 file
RAW_FILE = RAW_DIR / "HIL-005.json"

# Load the raw data
with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Loaded:", RAW_FILE.name)
print("Top-level type:", type(raw_data).__name__)

Loaded: HIL-005.json
Top-level type: dict


In [5]:
# Inspect top-level keys and their value types
for key, value in raw_data.items():
    print(f"{key}: {type(value).__name__}")

displayFieldName: str
fieldAliases: dict
geometryType: str
spatialReference: dict
fields: list
exceededTransferLimit: bool
features: list


In [6]:
import pandas as pd

# Extract feature attributes into a DataFrame
df = pd.DataFrame(
    [feature["attributes"] for feature in raw_data["features"]]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 43686
Columns: 26


In [7]:
# Consider null values and data types for each column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43686 entries, 0 to 43685
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   OBJECTID           43686 non-null  int64  
 1   BLDG_ID            43686 non-null  int64  
 2   STATUS             43686 non-null  str    
 3   NUM_STORIES        0 non-null      object 
 4   HEIGHT             39225 non-null  float64
 5   YEAR_BUILT         42591 non-null  float64
 6   SOURCE             43686 non-null  str    
 7   PLANREFID          5114 non-null   str    
 8   CONST_TYPE         0 non-null      object 
 9   ROOF_TYPE          71 non-null     str    
 10  ROOF_COVER         0 non-null      object 
 11  BASEMENT           0 non-null      object 
 12  SPRINKLED          0 non-null      object 
 13  NFIRSCD            0 non-null      object 
 14  OCCUPANCY_TYPE     0 non-null      object 
 15  Tracking_CreateID  43686 non-null  str    
 16  UTC_CreateDate     43686 non-null

In [8]:
# Distinguishing between missing values and empty values
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_percent"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
CONST_TYPE,43686,100.000000
NUM_STORIES,43686,100.000000
NFIRSCD,43686,100.000000
SPRINKLED,43686,100.000000
YEAR_DEMOLISHED,43686,100.000000
OCCUPANCY_TYPE,43686,100.000000
BASEMENT,43686,100.000000
ROOF_COVER,43686,100.000000
DEMO_PERMIT,43676,99.977109
OMS_FACILITY_ID,43624,99.858078


In [9]:
df["SOURCE"].value_counts(dropna=False)

SOURCE
3DiJUL1999         20445
DOGAMI2014          2883
iTENJUL2011         2881
OSIJUL2003          2867
PIXJUL2006          2565
iTENJUL2013         1722
PIXJUL2007          1432
SANJUN2008          1279
PIXJUL2004          1037
GEOTERRA2017         710
GEOTERRA2020         605
GEOTERRA2021         567
GEOTERRAJUN2025      537
iTENJUL2010          527
GEOTERRA2022         497
GEOTERRA2023         436
GEOTERRA2019         353
SANJUN2009           347
GEOTERRAMAR2024      326
GEOTERRA2018         278
Site Plan            253
iTEN2015             245
GEOTERRA2016         226
iTENJUL2012          223
iTENMAR2012          188
GEOTERRAJUN2024      143
LiDAR2007            109
SBGJUL2001             3
SBGJUL2002             2
Name: count, dtype: int64

## `SOURCE` Assessment

The `SOURCE` field is fully populated across all 43,686 records and contains 29 distinct values. The observed values correspond to the coded values documented by the City of Hillsboro GIS layer.

The source values appear to encode the provenance and approximate date of the imagery or data source used to establish building information. The distribution is highly uneven, with `3DiJUL1999` accounting for approximately 47% of records.

### Cleaning Decision

No cleaning is currently required for the `SOURCE` field. The original source values will be preserved as provided by the City of Hillsboro GIS layer.

If human-readable source descriptions or temporal analysis are needed later, those should be added as derived fields rather than replacing the original values.

In [10]:
df["YEAR_BUILT"].describe()

count    42591.000000
mean      1578.907915
std        802.128265
min          0.000000
25%       1945.000000
50%       1980.000000
75%       1999.000000
max       2029.000000
Name: YEAR_BUILT, dtype: float64

In [11]:
# The mean is likely reduced from the presence of "0" values, and that the max is greater than the current year
print("YEAR_BUILT = 0:", (df["YEAR_BUILT"] == 0).sum())
print("YEAR_BUILT > 2026:", (df["YEAR_BUILT"] > 2026).sum())

YEAR_BUILT = 0: 8732
YEAR_BUILT > 2026: 1


In [12]:
# What does the building with a YEAR_BUILT greater than 2026 look like? Are there any other anomalies in the data?
df.loc[df["YEAR_BUILT"] > 2026]

,OBJECTID,BLDG_ID,STATUS,NUM_STORIES,HEIGHT,YEAR_BUILT,SOURCE,PLANREFID,CONST_TYPE,ROOF_TYPE,...,UTC_CreateDate,Tracking_EditID,UTC_EditDate,GlobalID,YEAR_DEMOLISHED,DEMO_PERMIT,Shape.STArea(),Shape.STLength(),OMS_FACILITY_ID,PERMIT_ID
43572,103584,111554,0,None,NaN,2029.0,Site Plan,NaN,None,NaN,...,1755643693000,CLAAM,1768609754000,{D6BF79EA-6C60-43AA-859D-F9E3D5DD5FBD},None,NaN,928.535987,151.699769,NaN,CMB25-00176


## `YEAR_BUILT` Anomaly Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0` and one record with a value of `2029`.

The 2029 record has:
- `STATUS = 0` (Active)
- `SOURCE = Site Plan`
- `PERMIT_ID = CMB25-00176`

The combination of `SOURCE = Site Plan` and the presence of a building permit identifier provides context for the future year, but does not establish why `YEAR_BUILT` is recorded as 2029.

### Cleaning Decision

The 2029 value will be preserved. It will be treated as an anomalous value for quality-control purposes rather than automatically classified as erroneous or replaced with a null value.

In [13]:
# What are the sources of the buildings with a YEAR_BUILT of 0? Are there any other anomalies in the data?
# Start by looking at the sources of the buildings with a YEAR_BUILT of 0
df.loc[df["YEAR_BUILT"] == 0, "SOURCE"].value_counts()

SOURCE
3DiJUL1999      3009
DOGAMI2014      1765
iTENJUL2011      844
iTENJUL2013      827
iTENJUL2010      362
OSIJUL2003       264
iTEN2015         241
GEOTERRA2016     226
PIXJUL2006       222
PIXJUL2004       210
iTENMAR2012      180
iTENJUL2012      145
SANJUN2008       144
PIXJUL2007       130
SANJUN2009        64
LiDAR2007         43
GEOTERRA2017      24
GEOTERRA2020       7
GEOTERRA2019       6
GEOTERRA2018       6
GEOTERRA2022       4
GEOTERRA2021       3
SBGJUL2001         3
SBGJUL2002         2
GEOTERRA2023       1
Name: count, dtype: int64

In [14]:
# Calculate the percentage of buildings with YEAR_BUILT = 0 for each source
zero_by_source = df["SOURCE"].where(df["YEAR_BUILT"] == 0).value_counts()
total_by_source = df["SOURCE"].value_counts()

zero_percent_by_source = (
    zero_by_source / total_by_source * 100
).sort_values(ascending=False)

zero_percent_by_source

SOURCE
GEOTERRA2016       100.000000
SBGJUL2001         100.000000
SBGJUL2002         100.000000
iTEN2015            98.367347
iTENMAR2012         95.744681
iTENJUL2010         68.690702
iTENJUL2012         65.022422
DOGAMI2014          61.220950
iTENJUL2013         48.025552
LiDAR2007           39.449541
iTENJUL2011         29.295384
PIXJUL2004          20.250723
SANJUN2009          18.443804
3DiJUL1999          14.717535
SANJUN2008          11.258796
OSIJUL2003           9.208232
PIXJUL2007           9.078212
PIXJUL2006           8.654971
GEOTERRA2017         3.380282
GEOTERRA2018         2.158273
GEOTERRA2019         1.699717
GEOTERRA2020         1.157025
GEOTERRA2022         0.804829
GEOTERRA2021         0.529101
GEOTERRA2023         0.229358
GEOTERRAJUN2024           NaN
GEOTERRAJUN2025           NaN
GEOTERRAMAR2024           NaN
Site Plan                 NaN
Name: count, dtype: float64

## `YEAR_BUILT` Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0`, representing approximately 20% of the dataset. The value occurs across 25 of the 29 observed `SOURCE` values, but its prevalence varies substantially by source.

Some sources contain `YEAR_BUILT = 0` for nearly all records, while others contain very few or no zero values. This indicates that the use or availability of construction-year information varies by source.

### Cleaning Decision

No values in `YEAR_BUILT` will be modified at this stage. The original values, including `0` and `2029`, will be preserved while the meaning of the zero-value convention is investigated further.

## Cleaning: Convert Date Fields

The source schema identifies `UTC_CreateDate` and `UTC_EditDate` as date
fields. The downloaded JSON represents these values as Unix epoch
timestamps in milliseconds.

These fields will be converted to pandas datetime values while preserving
their UTC interpretation.

In [15]:
# Convert the UTC_CreateDate and UTC_EditDate columns to datetime objects
date_columns = [
    "UTC_CreateDate",
    "UTC_EditDate"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        unit="ms",
        utc=True
    )

# Inspect the date columns after conversion
df[date_columns].info()
df[date_columns].head()

<class 'pandas.DataFrame'>
RangeIndex: 43686 entries, 0 to 43685
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   UTC_CreateDate  43686 non-null  datetime64[ms, UTC]
 1   UTC_EditDate    43686 non-null  datetime64[ms, UTC]
dtypes: datetime64[ms, UTC](2)
memory usage: 682.7 KB


,UTC_CreateDate,UTC_EditDate
0,2019-02-04 23:40:40+00:00,2026-08-11 20:12:31+00:00
1,2019-02-04 23:40:40+00:00,2020-01-06 22:50:48+00:00
2,2019-02-04 23:40:40+00:00,2019-02-05 00:11:55+00:00
3,2019-02-04 23:40:40+00:00,2019-03-11 20:58:48+00:00
4,2019-02-04 23:40:40+00:00,2021-03-05 19:54:40+00:00


In [16]:
# Inspect the min and max values of the date columns to check for anomalies
df[date_columns].agg(["min", "max"])

,UTC_CreateDate,UTC_EditDate
min,2019-02-04 23:40:40+00:00,2019-02-05 00:11:55+00:00
max,2026-08-12 23:02:55+00:00,2026-08-12 23:02:55+00:00


## Cleaning: YEAR_BUILT

Both CreateDate and EditDates look reasonable after transformation, and show no need for further manipulation.

However, YEAR_BUILT has shown a few anomalies and will likely need more in-depth cleaning.

`YEAR_BUILT` contains 8,732 records with a value of `0`. Because `0` is
not a valid construction year, these values are treated as unknown rather
than literal years.

One record contains `YEAR_BUILT = 2029`, which is beyond the current
calendar year but will remain in the dataset.

### Cleaning Rule

- Replace `YEAR_BUILT = 0` with a missing value.
- Retain `YEAR_BUILT = 2029` for now.
- Do not discard building records based solely on an unusual construction
  year.

In [17]:
# Convert YEAR_BUILT values of 0 to NaN to avoid skewing the mean and other statistics
df["YEAR_BUILT"] = df["YEAR_BUILT"].replace(0, pd.NA)

print("YEAR_BUILT missing:", df["YEAR_BUILT"].isna().sum())

YEAR_BUILT missing: 9827


In [18]:
# With the YEAR_BUILT values of 0 replaced with NaN, we can now check the data type of the 
# YEAR_BUILT column to ensure it is appropriate for further analysis.
df["YEAR_BUILT"].dtype

dtype('O')

In [19]:
# Convert YEAR_BUILT to a numeric type, coercing errors to NaN, and then convert to Int64 to allow for missing values
df["YEAR_BUILT"] = pd.to_numeric(
    df["YEAR_BUILT"],
    errors="coerce"
).astype("Int64")

In [20]:
df["YEAR_BUILT"].dtype

Int64Dtype()

In [21]:
# Lastly, check the number of missing values in the YEAR_BUILT column after the conversion to Int64
print("YEAR_BUILT missing:", df["YEAR_BUILT"].isna().sum())

YEAR_BUILT missing: 9827


In [22]:
# Check the data type, number of missing values, and min/max values of the YEAR_BUILT column
print("Dtype:", df["YEAR_BUILT"].dtype)
print("Missing:", df["YEAR_BUILT"].isna().sum())
print("Minimum:", df["YEAR_BUILT"].min())
print("Maximum:", df["YEAR_BUILT"].max())

Dtype: Int64
Missing: 9827
Minimum: 1001
Maximum: 2029


### Validation Note

After cleaning, `YEAR_BUILT` contains 9,827 missing values and ranges from
1001 to 2029. The minimum value of 1001 is anomalous for this dataset and
should be investigated separately rather than automatically removed.

## Cleaning: STATUS

The source defines `STATUS` as a coded domain:

- `0` = Active
- `1` = Demoed
- `2` = Permitted

The raw coded value will be preserved. A new `STATUS_LABEL` field will be
created to provide a human-readable representation.

In [23]:
status_labels = {
    "0": "Active",
    "1": "Demoed",
    "2": "Permitted"
}

df["STATUS_LABEL"] = df["STATUS"].map(status_labels)

In [24]:
df[["STATUS", "STATUS_LABEL"]].value_counts()

STATUS  STATUS_LABEL
0       Active          43686
Name: count, dtype: int64

In [25]:
# Verify that the new mapping of STATUS to STATUS_LABEL is correct by checking the number of missing values in both columns
print("STATUS missing:", df["STATUS"].isna().sum())
print("STATUS_LABEL missing:", df["STATUS_LABEL"].isna().sum())

STATUS missing: 0
STATUS_LABEL missing: 0


In [26]:
df[["STATUS", "STATUS_LABEL"]].drop_duplicates()

,STATUS,STATUS_LABEL
0,0,Active


## Data Quality Review: Field Cardinality

Before applying additional cleaning rules, examine the number of unique
values in each field. Fields with very few unique values may represent
coded domains or categorical information, while fields with little or no
variation may provide limited analytical value.

In [27]:
df.nunique(dropna=False).sort_values()

NUM_STORIES              1
STATUS                   1
NFIRSCD                  1
OCCUPANCY_TYPE           1
SPRINKLED                1
BASEMENT                 1
ROOF_COVER               1
CONST_TYPE               1
STATUS_LABEL             1
YEAR_DEMOLISHED          1
Tracking_CreateID        2
DEMO_PERMIT              3
Tracking_EditID          3
ROOF_TYPE                4
SOURCE                  29
OMS_FACILITY_ID         49
YEAR_BUILT             129
PLANREFID              363
UTC_EditDate          2422
UTC_CreateDate        2599
PERMIT_ID             3234
HEIGHT               23744
Shape.STLength()     43545
Shape.STArea()       43574
OBJECTID             43686
BLDG_ID              43686
GlobalID             43686
dtype: int64

## Field Classification

Each field is classified according to its role in the source dataset.
Classification will guide subsequent cleaning decisions while preserving
source information unless there is a documented reason to transform or
remove it.

Categories include:

- Identifier
- Categorical
- Numeric
- Date
- Provenance / Metadata
- Geometry-derived
- Completely Missing

In [28]:
field_classification = {
    "OBJECTID": "Identifier",
    "BLDG_ID": "Identifier",
    "GlobalID": "Identifier",
    
    "STATUS": "Categorical",
    "STATUS_LABEL": "Categorical",
    "SOURCE": "Provenance / Metadata",
    "ROOF_TYPE": "Categorical",
    "DEMO_PERMIT": "Identifier",
    "PLANREFID": "Identifier",
    "PERMIT_ID": "Identifier",
    "OMS_FACILITY_ID": "Identifier",
    
    "NUM_STORIES": "Numeric",
    "HEIGHT": "Numeric",
    "YEAR_BUILT": "Numeric",
    "YEAR_DEMOLISHED": "Numeric",
    
    "UTC_CreateDate": "Date",
    "UTC_EditDate": "Date",
    
    "Tracking_CreateID": "Provenance / Metadata",
    "Tracking_EditID": "Provenance / Metadata",
    
    "CONST_TYPE": "Completely Missing",
    "ROOF_COVER": "Completely Missing",
    "BASEMENT": "Completely Missing",
    "SPRINKLED": "Completely Missing",
    "NFIRSCD": "Completely Missing",
    "OCCUPANCY_TYPE": "Completely Missing",
    
    "Shape.STArea()": "Geometry-derived",
    "Shape.STLength()": "Geometry-derived",
}

In [29]:
classification = pd.Series(field_classification, name="classification")

classification

OBJECTID                        Identifier
BLDG_ID                         Identifier
GlobalID                        Identifier
STATUS                         Categorical
STATUS_LABEL                   Categorical
SOURCE               Provenance / Metadata
ROOF_TYPE                      Categorical
DEMO_PERMIT                     Identifier
PLANREFID                       Identifier
PERMIT_ID                       Identifier
OMS_FACILITY_ID                 Identifier
NUM_STORIES                        Numeric
HEIGHT                             Numeric
YEAR_BUILT                         Numeric
YEAR_DEMOLISHED                    Numeric
UTC_CreateDate                        Date
UTC_EditDate                          Date
Tracking_CreateID    Provenance / Metadata
Tracking_EditID      Provenance / Metadata
CONST_TYPE              Completely Missing
ROOF_COVER              Completely Missing
BASEMENT                Completely Missing
SPRINKLED               Completely Missing
NFIRSCD    

In [30]:
# Table describing the fields in the DataFrame, their classifications, data types, number of non-null values, and number of unique values
field_audit = pd.DataFrame({
    "field": df.columns,
    "classification": [
        field_classification.get(field, "Unclassified")
        for field in df.columns
    ],
    "dtype": [
        str(df[field].dtype)
        for field in df.columns
    ],
    "non_null": [
        df[field].notna().sum()
        for field in df.columns
    ],
    "unique": [
        df[field].nunique(dropna=False)
        for field in df.columns
    ]
})

field_audit

,field,classification,dtype,non_null,unique
0,OBJECTID,Identifier,int64,43686,43686
1,BLDG_ID,Identifier,int64,43686,43686
2,STATUS,Categorical,str,43686,1
3,NUM_STORIES,Numeric,object,0,1
4,HEIGHT,Numeric,float64,39225,23744
5,YEAR_BUILT,Numeric,Int64,33859,129
6,SOURCE,Provenance / Metadata,str,43686,29
7,PLANREFID,Identifier,str,5114,363
8,CONST_TYPE,Completely Missing,object,0,1
9,ROOF_TYPE,Categorical,str,71,4


In [31]:
# Verifies that all fields in the DataFrame have been classified, and identifies any unclassified fields
unclassified = [
    field for field in df.columns
    if field not in field_classification
]

unclassified

[]

## Retention Decision: Completely Missing Fields

Several fields contain no populated values in the current dataset:

- `NUM_STORIES`
- `NFIRSCD`
- `OCCUPANCY_TYPE`
- `SPRINKLED`
- `BASEMENT`
- `ROOF_COVER`
- `CONST_TYPE`
- `YEAR_DEMOLISHED`

These fields will be retained in the source-faithful dataset for
provenance and documentation purposes.

They are candidates for exclusion from a future analytical dataset if they
remain completely unpopulated.

No source fields will be removed during this initial cleaning stage solely
because they are empty.

In [32]:
# Checking for anomalies in the HEIGHT column, such as negative values or unusually high/low values
df["HEIGHT"].describe()

count    39225.000000
mean        11.702715
std          9.115906
min          0.000000
25%          0.000000
50%         11.579300
75%         18.312799
max         95.909142
Name: HEIGHT, dtype: float64

In [33]:
# Attain the number of buildings with a HEIGHT of 0, as well as the number of missing values in the HEIGHT column
height_zero = (df["HEIGHT"] == 0).sum()

print("HEIGHT = 0:", height_zero)
print("HEIGHT missing:", df["HEIGHT"].isna().sum())

HEIGHT = 0: 10996
HEIGHT missing: 4461


In [34]:
# Check to see if Heights of 0 are associated with a particular attribute
df.loc[
    df["HEIGHT"] == 0,
    ["BLDG_ID", "STATUS", "YEAR_BUILT", "SOURCE"]
].head(20)

,BLDG_ID,STATUS,YEAR_BUILT,SOURCE
11,63344,0,<NA>,PIXJUL2006
28,63362,0,2003,OSIJUL2003
29,63363,0,<NA>,PIXJUL2004
34,63370,0,<NA>,OSIJUL2003
37,63373,0,<NA>,SANJUN2009
38,63374,0,1994,PIXJUL2006
41,63377,0,2007,SANJUN2008
45,63381,0,<NA>,SANJUN2008
47,63383,0,<NA>,PIXJUL2006
53,63390,0,1959,PIXJUL2006


In [35]:
# Count the number of buildings with a HEIGHT of 0 for each source
df.loc[df["HEIGHT"] == 0, "SOURCE"].value_counts()

SOURCE
3DiJUL1999         3321
DOGAMI2014         1768
PIXJUL2006         1587
iTENJUL2013         882
PIXJUL2007          645
iTENJUL2011         562
SANJUN2008          400
iTENJUL2010         377
OSIJUL2003          297
PIXJUL2004          251
iTEN2015            236
GEOTERRA2016        224
iTENMAR2012         179
iTENJUL2012         131
SANJUN2009           96
GEOTERRA2017         14
LiDAR2007            10
GEOTERRA2020          4
GEOTERRA2019          3
GEOTERRA2022          3
GEOTERRA2018          3
GEOTERRA2021          2
GEOTERRAMAR2024       1
Name: count, dtype: int64

In [36]:
df.loc[df["HEIGHT"] > 80, ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE"]].sort_values(
    "HEIGHT",
    ascending=False
)

,BLDG_ID,HEIGHT,YEAR_BUILT,SOURCE
29798,94866,95.909142,2008,SANJUN2008
35274,100710,90.000000,<NA>,iTENJUL2010


### Validation Findings

`HEIGHT` contains 10,996 records with a value of `0`. These records are
distributed across numerous source datasets rather than being isolated to a
single source.

Because a zero-foot building height is unlikely to represent a literal
building height, these values are considered potential missing/sentinel
values. The source documentation should be consulted before converting them
to missing values.

The maximum observed height is 95.91 feet. Only two records exceed 80 feet,
with heights of 95.91 and 90.00 feet. These values are not considered
anomalous based on the available evidence and will be retained.

## Investigation: Large Buildings

Two buildings were identified as having heights greater than 80 feet.
Their `BLDG_ID` values will be cross-referenced against the City's address
layer using `BuildingID`.

This is an exploratory lookup only and does not modify the HIL-005 dataset.

In [37]:
import requests

In [38]:
address_url = (
    "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
    "PW_FiberAddresses/MapServer/0/query"
)

params = {
    "where": "BuildingID IN (94866, 100710)",
    "outFields": "BuildingID,FullAddress,SubPudComplexName,X_LONG,Y_LAT",
    "returnGeometry": "false",
    "f": "json"
}

response = requests.get(address_url, params=params)
response.raise_for_status()

address_data = response.json()

address_data

{'displayFieldName': 'SiteAddress',
 'fieldAliases': {'BuildingID': 'BuildingID',
  'FullAddress': 'Full Address',
  'SubPudComplexName': 'Sub PUD Complex Name',
  'X_LONG': 'X_LONG',
  'Y_LAT': 'Y_LAT'},
 'fields': [{'name': 'BuildingID',
   'type': 'esriFieldTypeInteger',
   'alias': 'BuildingID'},
  {'name': 'FullAddress',
   'type': 'esriFieldTypeString',
   'alias': 'Full Address',
   'length': 250},
  {'name': 'SubPudComplexName',
   'type': 'esriFieldTypeString',
   'alias': 'Sub PUD Complex Name',
   'length': 100},
  {'name': 'X_LONG', 'type': 'esriFieldTypeDouble', 'alias': 'X_LONG'},
  {'name': 'Y_LAT', 'type': 'esriFieldTypeDouble', 'alias': 'Y_LAT'}],
 'features': [{'attributes': {'BuildingID': 94866,
    'FullAddress': '1862 NW 9TH AVE, HILLSBORO, OR 97124',
    'SubPudComplexName': 'UNKNOWN',
    'X_LONG': -122.99794163,
    'Y_LAT': 45.53854491}},
  {'attributes': {'BuildingID': 100710,
    'FullAddress': '4573 SE SATINWOOD ST, HILLSBORO, OR 97123',
    'SubPudComplexNa

In [39]:
for feature in address_data["features"]:
    print(feature["attributes"])

{'BuildingID': 94866, 'FullAddress': '1862 NW 9TH AVE, HILLSBORO, OR 97124', 'SubPudComplexName': 'UNKNOWN', 'X_LONG': -122.99794163, 'Y_LAT': 45.53854491}
{'BuildingID': 100710, 'FullAddress': '4573 SE SATINWOOD ST, HILLSBORO, OR 97123', 'SubPudComplexName': 'BROOKWOOD CROSSING NO. 5', 'X_LONG': -122.93632363, 'Y_LAT': 45.49744841}
{'BuildingID': 100710, 'FullAddress': '4573 SE SATINWOOD ST, HILLSBORO, OR 97129', 'SubPudComplexName': 'BROOKWOOD CROSSING NO. 5', 'X_LONG': -122.93632245, 'Y_LAT': 45.49742099}


In [40]:
df.loc[
    df["BLDG_ID"].isin([94866, 100710]),
    ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE", "Shape.STArea()", "Shape.STLength()"]
]

,BLDG_ID,HEIGHT,YEAR_BUILT,SOURCE,Shape.STArea(),Shape.STLength()
29798,94866,95.909142,2008,SANJUN2008,1767.733839,172.221878
35274,100710,90.000000,<NA>,iTENJUL2010,1210.466599,150.406930


### Investigation Findings

Two records exceeded 80 feet in height:

- `BLDG_ID 94866`: 95.91 feet
- `BLDG_ID 100710`: 90.00 feet

Both records were cross-referenced against the City's address layer and
visually inspected using external mapping imagery. The associated locations
appear to be residential structures rather than buildings approaching
90–96 feet in height.

Their building footprints are also relatively modest:

- `BLDG_ID 94866`: 1,767.7 square feet
- `BLDG_ID 100710`: 1,210.5 square feet

These observations provide strong evidence that the recorded heights may be
erroneous. The records will be flagged for further investigation rather than
silently corrected, since the underlying cause of the anomalous values has
not yet been established.

In [41]:
from pathlib import Path
import json

project_root = Path.cwd().parent
DATA_VERSION = "2026-08-26"

raw_path = project_root / DATA_VERSION / "raw" / "HIL-005.json"

with open(raw_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded: {raw_path}")
print(f"Features: {len(data['features']):,}")

Loaded: c:\Users\John\Documents\hillsborogis\2026-08-26\raw\HIL-005.json
Features: 43,686


In [42]:
# Verifying the address of the buildings with BLDG_IDs 94866 and 100710 by checking the address data returned from the GIS service
target_ids = {94866, 100710}

target_features = [
    feature
    for feature in data["features"]
    if feature["attributes"]["BLDG_ID"] in target_ids
]

print(f"Found: {len(target_features)} features")

for feature in target_features:
    attrs = feature["attributes"]
    print(
        attrs["BLDG_ID"],
        attrs["HEIGHT"],
        attrs["Shape.STArea()"],
        feature["geometry"]["rings"][0][:2]
    )

Found: 2 features
94866 95.90914154 1767.7338393878176 [[-13692067.913720848, 5706695.203891196], [-13692083.166896138, 5706709.399740975]]
100710 90.0 1210.46659920924 [[-13685213.059714176, 5700165.496016747], [-13685213.130361034, 5700188.0869342685]]


In [43]:
data["spatialReference"]

{'wkid': 102100, 'latestWkid': 3857}

In [44]:
from pyproj import Transformer

In [45]:
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)

In [46]:
address_points = {
    94866: (-122.99794163, 45.53854491),
    100710: (-122.93632363, 45.49744841),
}

projected_points = {}

for building_id, (lon, lat) in address_points.items():
    x, y = transformer.transform(lon, lat)
    projected_points[building_id] = (x, y)
    print(building_id, x, y)

94866 -13692068.230872385 5706706.591320579
100710 -13685208.946488684 5700177.494125503


In [47]:
from shapely.geometry import Point, Polygon

In [48]:
from shapely.geometry import Polygon

building_polygons = {}

for feature in target_features:
    attrs = feature["attributes"]
    building_id = attrs["BLDG_ID"]
    ring = feature["geometry"]["rings"][0]
    
    building_polygons[building_id] = Polygon(ring)

for building_id, polygon in building_polygons.items():
    print(
        f"BLDG_ID {building_id}: "
        f"valid={polygon.is_valid}, "
        f"area={polygon.area:.2f}"
    )

BLDG_ID 94866: valid=True, area=334.77
BLDG_ID 100710: valid=True, area=228.90


## Height Investigation — Polygon Validation

The two buildings identified as height anomalies (`BLDG_ID` 94866 and `BLDG_ID` 100710) were retrieved from the original HIL-005 geometry and converted to Shapely polygons.

Both reconstructed polygons are geometrically valid. However, the planar areas calculated by Shapely do not match the `Shape.STArea()` values provided by the ArcGIS dataset:

- `BLDG_ID 94866`: Shapely area = 334.77 vs. `Shape.STArea()` = 1,767.73
- `BLDG_ID 100710`: Shapely area = 228.90 vs. `Shape.STArea()` = 1,210.47

This discrepancy does not necessarily indicate an error in the geometry. ArcGIS and Shapely may calculate or interpret geometry measurements differently, particularly with projected GIS data.

For the current investigation, the area discrepancy does not prevent us from testing the proposed relationship between `BLDG_ID` and the address-layer `BuildingID`. The next step is to compare the projected address coordinates with the actual building polygon locations.

**Finding:** The polygon geometries are valid, but their calculated areas require additional interpretation before using area measurements for QA or analysis.

In [49]:
for building_id, polygon in building_polygons.items():
    print(f"\nBLDG_ID {building_id}")
    print(f"Bounds: {polygon.bounds}")
    print(f"Area: {polygon.area:.2f}")


BLDG_ID 94866
Bounds: (-13692083.166896138, 5706695.203891196, -13692056.989877649, 5706721.501842046)
Area: 334.77

BLDG_ID 100710
Bounds: (-13685213.130361034, 5700165.496016747, -13685202.927394398, 5700188.118758175)
Area: 228.90


In [50]:
from shapely.geometry import Point

for building_id, point_coords in projected_points.items():
    polygon = building_polygons[building_id]
    point = Point(point_coords)

    print(
        f"BLDG_ID {building_id}: "
        f"inside={polygon.contains(point)}, "
        f"distance={polygon.distance(point):.2f} meters"
    )

BLDG_ID 94866: inside=True, distance=0.00 meters
BLDG_ID 100710: inside=True, distance=0.00 meters


## Height Investigation — Building ID Relationship Validation

The relationship between the HIL-005 `BLDG_ID` field and the `BuildingID` field used in the address lookup was validated spatially for the two buildings identified as height anomalies.

The address coordinates were originally returned in WGS 84 (EPSG:4326) and were projected to the HIL-005 spatial reference (EPSG:3857 / WKID 102100). The resulting address points were then compared with the corresponding HIL-005 building polygons.

Results:

- `BLDG_ID 94866`: address point falls inside the HIL-005 building polygon.
- `BLDG_ID 100710`: address point falls inside the HIL-005 building polygon.

Both tests returned `inside=True` and a polygon distance of `0.00` meters.

**Finding:** The matching `BLDG_ID` / `BuildingID` relationship is supported by spatial evidence for both investigated records. Using the building ID to retrieve their addresses was therefore a reasonable method for investigating the anomalous height values.

This validation applies specifically to the records tested and should not automatically be interpreted as proof that the relationship is valid for every record in the two datasets.

In [51]:
# Flag unusually tall buildings for further review
df["HEIGHT_ANOMALY"] = df["HEIGHT"] > 80

print(df["HEIGHT_ANOMALY"].value_counts())

HEIGHT_ANOMALY
False    43684
True         2
Name: count, dtype: int64


In [52]:
df.loc[
    df["HEIGHT_ANOMALY"],
    ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE"]
]

,BLDG_ID,HEIGHT,YEAR_BUILT,SOURCE
29798,94866,95.909142,2008,SANJUN2008
35274,100710,90.000000,<NA>,iTENJUL2010


## Cleaning: HEIGHT

`HEIGHT` was reviewed for missing values, zero values, unusually large values, and potential anomalies.

- 4,461 records have a missing `HEIGHT`.
- 10,996 records contain `HEIGHT = 0`.
- Two records contain heights greater than 80 feet:
  - `BLDG_ID 94866`: 95.91 feet
  - `BLDG_ID 100710`: 90.00 feet
- Both records were investigated using the associated building ID and an independent address dataset.
- The `BLDG_ID` / `BuildingID` relationship was spatially validated: the retrieved address points fall within the corresponding HIL-005 building polygons.
- No authoritative source was identified that establishes replacement height values for these records.

**Decision:** Retain the original height values and flag the two records as height anomalies rather than altering or deleting potentially valid source data.

The `HEIGHT_ANOMALY` field is an analytical flag and does not replace the source `HEIGHT` value.

## Cleaning: SOURCE — Initial Review

The `SOURCE` field identifies the apparent source or origin of each building record. Unlike fields such as `HEIGHT` or `YEAR_BUILT`, `SOURCE` is not simply a value to validate numerically; it is a provenance field and should be preserved carefully.

The initial review will focus on:

1. **Completeness** — determine whether any records lack a `SOURCE` value.
2. **Consistency** — identify distinct values and check for unexpected variations in capitalization, spacing, or formatting.
3. **Frequency** — determine how many building records are associated with each source.
4. **Meaning** — investigate what the source codes represent, including embedded dates or abbreviations.
5. **Temporal context** — compare source dates with `YEAR_BUILT` and other relevant fields where appropriate.
6. **Standardization** — determine whether the source values can be represented in a more useful, consistent form without losing the original provenance information.

The original `SOURCE` values will be retained. Any standardized or derived provenance fields will be created separately so that the source data remains traceable.

The goal is not to make every source value look uniform simply for the sake of consistency. Instead, the goal is to make the provenance information easier to interpret while preserving what the original dataset tells us.

In [53]:
# Check how many building records are missing a SOURCE value.
# This tells us whether the provenance field is complete.
print(f"SOURCE missing: {df['SOURCE'].isna().sum():,}")

SOURCE missing: 0


In [54]:
# Display the distinct SOURCE values in alphabetical order.
# This makes it easier to identify naming patterns such as prefixes, dates, and
# inconsistent formatting before we decide whether any standardization is needed.
print(sorted(df["SOURCE"].unique()))

['3DiJUL1999', 'DOGAMI2014', 'GEOTERRA2016', 'GEOTERRA2017', 'GEOTERRA2018', 'GEOTERRA2019', 'GEOTERRA2020', 'GEOTERRA2021', 'GEOTERRA2022', 'GEOTERRA2023', 'GEOTERRAJUN2024', 'GEOTERRAJUN2025', 'GEOTERRAMAR2024', 'LiDAR2007', 'OSIJUL2003', 'PIXJUL2004', 'PIXJUL2006', 'PIXJUL2007', 'SANJUN2008', 'SANJUN2009', 'SBGJUL2001', 'SBGJUL2002', 'Site Plan', 'iTEN2015', 'iTENJUL2010', 'iTENJUL2011', 'iTENJUL2012', 'iTENJUL2013', 'iTENMAR2012']


In [55]:
# Extract a four-digit year from each SOURCE value where a year is present.
# This lets us separate the source's apparent date from the original SOURCE label
# without modifying the original provenance field.
df["SOURCE_YEAR"] = (
    df["SOURCE"]
    .str.extract(r"(\d{4})", expand=False)
    .astype("Int64")
)

# Display the distinct SOURCE values alongside their extracted years.
# This helps us verify that the extraction works across the different naming patterns.
print(
    df[["SOURCE", "SOURCE_YEAR"]]
    .drop_duplicates()
    .sort_values(["SOURCE_YEAR", "SOURCE"])
    .to_string(index=False)
)

         SOURCE  SOURCE_YEAR
     3DiJUL1999         1999
     SBGJUL2001         2001
     SBGJUL2002         2002
     OSIJUL2003         2003
     PIXJUL2004         2004
     PIXJUL2006         2006
      LiDAR2007         2007
     PIXJUL2007         2007
     SANJUN2008         2008
     SANJUN2009         2009
    iTENJUL2010         2010
    iTENJUL2011         2011
    iTENJUL2012         2012
    iTENMAR2012         2012
    iTENJUL2013         2013
     DOGAMI2014         2014
       iTEN2015         2015
   GEOTERRA2016         2016
   GEOTERRA2017         2017
   GEOTERRA2018         2018
   GEOTERRA2019         2019
   GEOTERRA2020         2020
   GEOTERRA2021         2021
   GEOTERRA2022         2022
   GEOTERRA2023         2023
GEOTERRAJUN2024         2024
GEOTERRAMAR2024         2024
GEOTERRAJUN2025         2025
      Site Plan         <NA>


In [56]:
# Identify records where the apparent SOURCE year is later than YEAR_BUILT.
# A source created after construction is normal, so this is not an anomaly by itself.
# Instead, we are looking for the opposite relationship: a source dated before the
# recorded construction year, which could indicate a questionable YEAR_BUILT value.
source_before_build = df[
    df["SOURCE_YEAR"].notna()
    & df["YEAR_BUILT"].notna()
    & (df["SOURCE_YEAR"] < df["YEAR_BUILT"])
]

print(f"Records where SOURCE_YEAR < YEAR_BUILT: {len(source_before_build):,}")

Records where SOURCE_YEAR < YEAR_BUILT: 421


In [57]:
# Display the records where the apparent source date precedes the recorded
# construction year so we can determine whether the relationship is plausible
# or indicates another data-quality issue.
source_before_build[
    ["BLDG_ID", "YEAR_BUILT", "SOURCE", "SOURCE_YEAR"]
].head(20)

,BLDG_ID,YEAR_BUILT,SOURCE,SOURCE_YEAR
839,64188,2007,OSIJUL2003,2003
1053,64405,2005,OSIJUL2003,2003
1206,64561,2004,OSIJUL2003,2003
1482,64841,2005,OSIJUL2003,2003
2316,65694,2006,OSIJUL2003,2003
2333,65711,2004,OSIJUL2003,2003
2846,66237,2002,3DiJUL1999,1999
3333,66741,2003,3DiJUL1999,1999
3476,66897,2000,3DiJUL1999,1999
3670,67108,2002,3DiJUL1999,1999


In [58]:
# Count how many records have a SOURCE_YEAR earlier than YEAR_BUILT.
# This tells us whether the apparent temporal mismatch is an isolated issue
# or a common characteristic of the building dataset.
print(f"Records where SOURCE_YEAR < YEAR_BUILT: {len(source_before_build):,}")
print(
    f"Percentage of dataset: "
    f"{len(source_before_build) / len(df) * 100:.2f}%"
)

# Count the source values among records where SOURCE_YEAR precedes YEAR_BUILT.
# This helps determine whether the pattern is concentrated in particular source datasets.
print(source_before_build["SOURCE"].value_counts())

Records where SOURCE_YEAR < YEAR_BUILT: 421
Percentage of dataset: 0.96%
SOURCE
3DiJUL1999         126
GEOTERRA2019       109
GEOTERRA2021        37
GEOTERRAJUN2025     34
GEOTERRA2023        24
GEOTERRA2022        20
GEOTERRAJUN2024     20
GEOTERRA2018        17
OSIJUL2003          13
GEOTERRA2020        10
GEOTERRAMAR2024      7
PIXJUL2004           2
PIXJUL2006           1
DOGAMI2014           1
Name: count, dtype: int64


## SOURCE — Temporal Cross-Check

The apparent year embedded in `SOURCE` was extracted into a derived `SOURCE_YEAR` field and compared with `YEAR_BUILT`.

There are 421 records (0.96% of the dataset) where `SOURCE_YEAR` is earlier than `YEAR_BUILT`.

These records are concentrated primarily among several older and newer source groups, including `3DiJUL1999` and multiple `GEOTERRA` sources.

This relationship does not necessarily indicate an error. A source year may describe the date of an imagery or source dataset rather than the date when the building record was created or when its `YEAR_BUILT` value was established. Therefore, a source predating the recorded construction year can be legitimate depending on how the source field was used in the City's data-maintenance process.

**Decision:** Do not modify `YEAR_BUILT` or `SOURCE` based solely on this comparison. `SOURCE_YEAR` will be treated as contextual provenance information rather than a validation constraint against `YEAR_BUILT`.

In [59]:
# Extract the source-family prefix by removing the optional month abbreviation
# and four-digit year from the end of each SOURCE value.
# The original SOURCE field is preserved unchanged for provenance.
df["SOURCE_PREFIX"] = (
    df["SOURCE"]
    .str.replace(r"(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)?\d{4}$", "", regex=True)
    .str.strip()
)

# Display each original SOURCE value alongside its extracted source family.
# This lets us verify that the parsing worked correctly across all naming patterns.
print(
    df[["SOURCE", "SOURCE_PREFIX"]]
    .drop_duplicates()
    .sort_values(["SOURCE_PREFIX", "SOURCE"])
    .to_string(index=False)
)

         SOURCE SOURCE_PREFIX
     3DiJUL1999           3Di
     DOGAMI2014        DOGAMI
   GEOTERRA2016      GEOTERRA
   GEOTERRA2017      GEOTERRA
   GEOTERRA2018      GEOTERRA
   GEOTERRA2019      GEOTERRA
   GEOTERRA2020      GEOTERRA
   GEOTERRA2021      GEOTERRA
   GEOTERRA2022      GEOTERRA
   GEOTERRA2023      GEOTERRA
GEOTERRAJUN2024      GEOTERRA
GEOTERRAJUN2025      GEOTERRA
GEOTERRAMAR2024      GEOTERRA
      LiDAR2007         LiDAR
     OSIJUL2003           OSI
     PIXJUL2004           PIX
     PIXJUL2006           PIX
     PIXJUL2007           PIX
     SANJUN2008           SAN
     SANJUN2009           SAN
     SBGJUL2001           SBG
     SBGJUL2002           SBG
      Site Plan     Site Plan
       iTEN2015          iTEN
    iTENJUL2010          iTEN
    iTENJUL2011          iTEN
    iTENJUL2012          iTEN
    iTENJUL2013          iTEN
    iTENMAR2012          iTEN


In [60]:
# Count building records by the extracted source family.
# This shows the overall contribution of each source family to the dataset.
source_prefix_counts = df["SOURCE_PREFIX"].value_counts()

print(source_prefix_counts)

SOURCE_PREFIX
3Di          20445
iTEN          5786
PIX           5034
GEOTERRA      4678
DOGAMI        2883
OSI           2867
SAN           1626
Site Plan      253
LiDAR          109
SBG              5
Name: count, dtype: int64


## SOURCE — Prefix Classification

The `SOURCE` field contains 29 distinct values. These values follow a consistent pattern in which many combine a source-family prefix with an apparent month and/or year.

A derived `SOURCE_PREFIX` field was created by removing the optional month abbreviation and four-digit year from the original `SOURCE` value. The original `SOURCE` field was retained unchanged.

The 29 source values resolve into 10 source families:

- `3Di` — 20,445 records
- `iTEN` — 5,786 records
- `PIX` — 5,034 records
- `GEOTERRA` — 4,678 records
- `DOGAMI` — 2,883 records
- `OSI` — 2,867 records
- `SAN` — 1,626 records
- `Site Plan` — 253 records
- `LiDAR` — 109 records
- `SBG` — 5 records

The extracted source families account for all 43,686 building records.

### Source Interpretation

Targeted research was conducted to determine what the source-family prefixes represent. Some interpretations can be supported by authoritative or dataset-specific evidence, while others could not be verified.

**Higher-confidence interpretations:**

- `GEOTERRA` — associated with GeoTerra's aerial mapping, GIS, and surveying work.
- `DOGAMI` — Oregon Department of Geology and Mineral Industries.
- `LiDAR` — identifies a LiDAR-derived source or data type.
- `Site Plan` — identifies a site-plan-derived source.

**Unverified interpretations:**

- `3Di`
- `iTEN`
- `PIX`
- `OSI`
- `SAN`
- `SBG`

Although organizations or technologies with names matching some of these prefixes can be found through external research, no sufficiently strong Hillsboro-specific evidence was identified to establish those interpretations as fact.

**Decision:** `SOURCE` will be preserved exactly as provided by the original dataset. `SOURCE_PREFIX` provides a standardized grouping mechanism, while `SOURCE_YEAR` provides the apparent year encoded in the source label. Organization-level interpretations will not be written into the cleaned dataset unless they can be independently verified.

This approach preserves the original provenance information while avoiding unsupported assumptions about legacy source codes.

In [61]:
# Profile structural, identifier, numeric, date, and geometry quality before further cleaning.
identifier_columns = [
    "OBJECTID",
    "BLDG_ID",
    "GlobalID",
]

identifier_audit = pd.DataFrame({
    "field": identifier_columns,
    "missing": [df[field].isna().sum() for field in identifier_columns],
    "blank": [
        df[field].astype("string").str.strip().eq("").sum()
        for field in identifier_columns
    ],
    "duplicate_values": [
        df[field].duplicated(keep=False).sum()
        for field in identifier_columns
    ],
})

numeric_audit = pd.DataFrame({
    "field": ["HEIGHT", "YEAR_BUILT", "Shape.STArea()", "Shape.STLength()"],
    "missing": [df[field].isna().sum() for field in ["HEIGHT", "YEAR_BUILT", "Shape.STArea()", "Shape.STLength()"]],
    "zero": [(df[field] == 0).sum() for field in ["HEIGHT", "YEAR_BUILT", "Shape.STArea()", "Shape.STLength()"]],
    "negative": [(df[field] < 0).sum() for field in ["HEIGHT", "YEAR_BUILT", "Shape.STArea()", "Shape.STLength()"]],
})

date_order_violations = (df["UTC_EditDate"] < df["UTC_CreateDate"]).sum()

geometry_audit = pd.DataFrame({
    "check": [
        "feature count matches attributes",
        "missing geometry",
        "missing rings",
        "empty rings",
        "multiple rings",
    ],
    "count": [
        len(raw_data["features"]) == len(df),
        sum(feature.get("geometry") is None for feature in raw_data["features"]),
        sum("rings" not in feature.get("geometry", {}) for feature in raw_data["features"]),
        sum(
            not feature.get("geometry", {}).get("rings")
            for feature in raw_data["features"]
        ),
        sum(
            len(feature.get("geometry", {}).get("rings", [])) > 1
            for feature in raw_data["features"]
        ),
    ],
})

print("Identifier audit")
display(identifier_audit)
print("Numeric audit")
display(numeric_audit)
print(f"Date order violations: {date_order_violations:,}")
print("Geometry audit")
display(geometry_audit)


Identifier audit


,field,missing,blank,duplicate_values
0,OBJECTID,0,0,0
1,BLDG_ID,0,0,0
2,GlobalID,0,0,0


Numeric audit


,field,missing,zero,negative
0,HEIGHT,4461,10996,0
1,YEAR_BUILT,9827,0,0
2,Shape.STArea(),0,0,0
3,Shape.STLength(),0,0,0


Date order violations: 0
Geometry audit


,check,count
0,feature count matches attributes,True
1,missing geometry,0
2,missing rings,0
3,empty rings,0
4,multiple rings,51


In [62]:
# Build an analytical view from the validated source-faithful DataFrame.
# The original HEIGHT value is preserved in HEIGHT_SOURCE; zero is treated as a missing sentinel.
analysis_df = df.copy()
analysis_df["HEIGHT_SOURCE"] = analysis_df["HEIGHT"]
analysis_df["HEIGHT_ZERO_FLAG"] = analysis_df["HEIGHT"].eq(0)
analysis_df["HEIGHT"] = analysis_df["HEIGHT"].mask(analysis_df["HEIGHT_ZERO_FLAG"])

analysis_df["YEAR_BUILT_EARLY_FLAG"] = (
    analysis_df["YEAR_BUILT"].notna()
    & (analysis_df["YEAR_BUILT"] < 1800)
)
analysis_df["YEAR_BUILT_FUTURE_FLAG"] = (
    analysis_df["YEAR_BUILT"].notna()
    & (analysis_df["YEAR_BUILT"] > pd.Timestamp.now(tz="UTC").year)
)

analysis_df["YEAR_BUILT_QA_FLAG"] = "OK"
analysis_df.loc[analysis_df["YEAR_BUILT"].isna(), "YEAR_BUILT_QA_FLAG"] = "Unknown"
analysis_df.loc[analysis_df["YEAR_BUILT_EARLY_FLAG"], "YEAR_BUILT_QA_FLAG"] = "Review: early year"
analysis_df.loc[analysis_df["YEAR_BUILT_FUTURE_FLAG"], "YEAR_BUILT_QA_FLAG"] = "Review: future year"

print(f"Analytical rows: {len(analysis_df):,}")
print(f"HEIGHT values converted from zero to missing: {analysis_df['HEIGHT_ZERO_FLAG'].sum():,}")
print("YEAR_BUILT QA flags")
display(analysis_df["YEAR_BUILT_QA_FLAG"].value_counts().rename_axis("flag").to_frame("records"))
print("Height QA by source")
display(
    analysis_df.groupby("SOURCE", dropna=False)
    .agg(
        records=("BLDG_ID", "size"),
        height_missing=("HEIGHT", lambda values: values.isna().sum()),
        height_zero=("HEIGHT_ZERO_FLAG", "sum"),
        height_anomaly=("HEIGHT_ANOMALY", "sum"),
    )
    .sort_values("height_zero", ascending=False)
    .head(10)
)


Analytical rows: 43,686
HEIGHT values converted from zero to missing: 10,996
YEAR_BUILT QA flags


,records
flag,
OK,33855
Unknown,9827
Review: early year,3
Review: future year,1


Height QA by source


,records,height_missing,height_zero,height_anomaly
SOURCE,,,,
3DiJUL1999,20445,3330,3321,0
DOGAMI2014,2883,1768,1768,0
PIXJUL2006,2565,1588,1587,0
iTENJUL2013,1722,882,882,0
PIXJUL2007,1432,645,645,0
iTENJUL2011,2881,562,562,0
SANJUN2008,1279,400,400,1
iTENJUL2010,527,379,377,1
OSIJUL2003,2867,297,297,0


## Analytical Cleaning Review

The preceding checks support a limited analytical cleaning step:

- Keep the source-faithful `df` unchanged apart from transformations already documented earlier in the notebook.
- In `analysis_df`, treat `HEIGHT = 0` as an unknown height and preserve the original value in `HEIGHT_SOURCE`.
- Preserve `YEAR_BUILT = 2029` and the three unusually early construction years, but expose them through QA flags.
- Preserve all records, identifiers, provenance fields, and geometry-derived measurements.
- Do not remove completely missing source fields; they remain useful for schema comparison and provenance.

The following validation checks confirm that these choices do not alter record identity or chronology.

In [63]:
# Inspect the small set of construction-year records requiring review.
year_review = analysis_df.loc[
    analysis_df["YEAR_BUILT_QA_FLAG"].str.startswith("Review"),
    [
        "BLDG_ID",
        "YEAR_BUILT",
        "YEAR_BUILT_QA_FLAG",
        "SOURCE",
        "SOURCE_YEAR",
        "PERMIT_ID",
        "STATUS_LABEL",
    ],
].sort_values("YEAR_BUILT")

display(year_review)

# Validate that analytical cleaning preserved the source record set.
assert len(analysis_df) == len(df) == len(raw_data["features"])
assert analysis_df["BLDG_ID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert analysis_df["SOURCE"].equals(df["SOURCE"])
assert analysis_df["HEIGHT_SOURCE"].equals(df["HEIGHT"])
assert analysis_df["HEIGHT"].dropna().ge(0).all()
assert (analysis_df["UTC_EditDate"] >= analysis_df["UTC_CreateDate"]).all()
assert analysis_df["HEIGHT_ZERO_FLAG"].eq(analysis_df["HEIGHT_SOURCE"].eq(0)).all()

print("Validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Unique BLDG_ID values: {analysis_df['BLDG_ID'].nunique():,}")
print(f"Analytical HEIGHT missing: {analysis_df['HEIGHT'].isna().sum():,}")


,BLDG_ID,YEAR_BUILT,YEAR_BUILT_QA_FLAG,SOURCE,SOURCE_YEAR,PERMIT_ID,STATUS_LABEL
27579,92388,1001,Review: early year,3DiJUL1999,1999,NaN,Active
20443,84735,1079,Review: early year,3DiJUL1999,1999,NaN,Active
32088,97390,1650,Review: early year,DOGAMI2014,2014,NaN,Active
43572,111554,2029,Review: future year,Site Plan,<NA>,CMB25-00176,Active


Validation passed
Rows preserved: 43,686
Unique BLDG_ID values: 43,686
Analytical HEIGHT missing: 15,457


## Additional Field Quality Review

The analytical view now needs a closer review of partially populated identifiers, categorical fields, permit relationships, and all feature geometries. These checks distinguish values that can be standardized from values that should only be flagged for review.

In [64]:
# Audit partially populated identifier fields without changing their source values.
optional_identifier_columns = [
    "PLANREFID",
    "PERMIT_ID",
    "DEMO_PERMIT",
    "OMS_FACILITY_ID",
]

identifier_quality = []
for field in optional_identifier_columns:
    values = df[field].astype("string")
    nonblank = values.notna() & values.str.strip().ne("")
    identifier_quality.append(
        {
            "field": field,
            "missing": int(values.isna().sum()),
            "blank_or_whitespace": int((values.notna() & ~nonblank).sum()),
            "nonblank": int(nonblank.sum()),
            "duplicate_nonblank_values": int(
                values[nonblank].duplicated(keep=False).sum()
            ),
            "distinct_nonblank": int(values[nonblank].nunique()),
        }
    )

identifier_quality = pd.DataFrame(identifier_quality)
display(identifier_quality)

print("Nonblank identifier values with duplicates")
for field in optional_identifier_columns:
    values = df[field].astype("string").str.strip()
    duplicates = values[values.notna() & values.ne("")]
    duplicates = duplicates[duplicates.duplicated(keep=False)]
    if not duplicates.empty:
        print(f"{field}: {duplicates.nunique()} distinct duplicated values")
        display(
            df.loc[
                df[field].astype("string").str.strip().isin(duplicates.unique()),
                ["BLDG_ID", field, "STATUS", "SOURCE"],
            ].head(20)
        )


,field,missing,blank_or_whitespace,nonblank,duplicate_nonblank_values,distinct_nonblank
0,PLANREFID,38572,0,5114,5009,362
1,PERMIT_ID,40393,0,3293,114,3233
2,DEMO_PERMIT,43676,5,5,5,1
3,OMS_FACILITY_ID,43624,1,61,15,47


Nonblank identifier values with duplicates
PLANREFID: 257 distinct duplicated values


,BLDG_ID,PLANREFID,STATUS,SOURCE
7,63339,WHBFD061,0,OSIJUL2003
9,63342,WHBFD274,0,OSIJUL2003
10,63343,WHBFD061,0,OSIJUL2003
13,63346,WHBFD274,0,OSIJUL2003
15,63348,WHBFD141,0,OSIJUL2003
16,63349,WHBFD454,0,SANJUN2009
17,63350,WHBFD274,0,OSIJUL2003
18,63351,WHBFD061,0,OSIJUL2003
21,63354,WHBFD274,0,OSIJUL2003
24,63358,WHBFD034,0,OSIJUL2003


PERMIT_ID: 54 distinct duplicated values


,BLDG_ID,PERMIT_ID,STATUS,SOURCE
40003,107235,CMB19-00118,0,GEOTERRA2019
40031,106387,CMB18-00171,0,GEOTERRA2019
40032,106535,CMB18-00257,0,GEOTERRA2019
40033,106547,CMB18-00257,0,GEOTERRA2019
40046,107236,CMB19-00117,0,GEOTERRA2019
40053,106386,CMB18-00171,0,GEOTERRA2019
40088,107237,CMB19-00114,0,GEOTERRA2019
40161,107238,CMB19-00115,0,GEOTERRA2019
40176,107239,CMB19-00116,0,GEOTERRA2019
40406,106978,CMB19-00254,0,GEOTERRA2020


DEMO_PERMIT: 1 distinct duplicated values


,BLDG_ID,DEMO_PERMIT,STATUS,SOURCE
4492,67983,DMO19-00007,0,3DiJUL1999
4527,68021,DMO19-00007,0,3DiJUL1999
4655,68156,DMO19-00007,0,3DiJUL1999
5618,69173,DMO19-00007,0,3DiJUL1999
5844,69414,DMO19-00007,0,3DiJUL1999


OMS_FACILITY_ID: 1 distinct duplicated values


,BLDG_ID,OMS_FACILITY_ID,STATUS,SOURCE
41753,108494,Water Treatment Plant,0,LiDAR2007
41754,108495,Water Treatment Plant,0,LiDAR2007
41755,108496,Water Treatment Plant,0,LiDAR2007
41756,108497,Water Treatment Plant,0,LiDAR2007
41757,108498,Water Treatment Plant,0,LiDAR2007
41758,108499,Water Treatment Plant,0,iTENJUL2010
41759,108500,Water Treatment Plant,0,LiDAR2007
41760,108502,Water Treatment Plant,0,LiDAR2007
41761,108503,Water Treatment Plant,0,iTENJUL2010
41762,108504,Water Treatment Plant,0,GEOTERRA2020


In [65]:
# Review categorical values and relationships between status and permit fields.
categorical_columns = ["STATUS", "STATUS_LABEL", "ROOF_TYPE"]
for field in categorical_columns:
    print(f"{field} values")
    display(
        df[field]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
        .rename_axis(field)
        .to_frame("records")
    )

analysis_df["ROOF_TYPE_CLEAN"] = (
    analysis_df["ROOF_TYPE"].astype("string").str.strip().replace("", pd.NA)
)
analysis_df["DEMO_PERMIT_CLEAN"] = (
    analysis_df["DEMO_PERMIT"].astype("string").str.strip().replace("", pd.NA)
)
analysis_df["PERMIT_ID_CLEAN"] = (
    analysis_df["PERMIT_ID"].astype("string").str.strip().replace("", pd.NA)
)
analysis_df["PLANREFID_CLEAN"] = (
    analysis_df["PLANREFID"].astype("string").str.strip().replace("", pd.NA)
)
analysis_df["OMS_FACILITY_ID_CLEAN"] = (
    analysis_df["OMS_FACILITY_ID"].astype("string").str.strip().replace("", pd.NA)
)

analysis_df["PERMIT_REVIEW_FLAG"] = (
    analysis_df["DEMO_PERMIT_CLEAN"].notna()
    | analysis_df["PERMIT_ID_CLEAN"].notna()
)
analysis_df["DEMO_PERMIT_REVIEW_FLAG"] = analysis_df["DEMO_PERMIT_CLEAN"].notna()

permit_review = analysis_df.loc[
    analysis_df["PERMIT_REVIEW_FLAG"],
    [
        "BLDG_ID",
        "STATUS",
        "STATUS_LABEL",
        "DEMO_PERMIT_CLEAN",
        "PERMIT_ID_CLEAN",
        "SOURCE",
    ],
]

print(f"Records with any permit identifier: {len(permit_review):,}")
print(f"Records with nonblank DEMO_PERMIT despite current status: {analysis_df['DEMO_PERMIT_REVIEW_FLAG'].sum():,}")
display(permit_review.head(20))


STATUS values


,records
STATUS,
0,43686


STATUS_LABEL values


,records
STATUS_LABEL,
Active,43686


ROOF_TYPE values


,records
ROOF_TYPE,
<NA>,43615
Pitched,38
,23
Flat,10


Records with any permit identifier: 3,298
Records with nonblank DEMO_PERMIT despite current status: 5


,BLDG_ID,STATUS,STATUS_LABEL,DEMO_PERMIT_CLEAN,PERMIT_ID_CLEAN,SOURCE
1833,65196,0,Active,<NA>,STR21-00741,GEOTERRA2022
4492,67983,0,Active,DMO19-00007,<NA>,3DiJUL1999
4527,68021,0,Active,DMO19-00007,<NA>,3DiJUL1999
4655,68156,0,Active,DMO19-00007,<NA>,3DiJUL1999
5618,69173,0,Active,DMO19-00007,<NA>,3DiJUL1999
5844,69414,0,Active,DMO19-00007,<NA>,3DiJUL1999
13942,77881,0,Active,<NA>,CMB25-00503,Site Plan
15586,79625,0,Active,<NA>,CMB19-00573,GEOTERRA2020
18915,83136,0,Active,<NA>,STR19-00862,GEOTERRA2022
19719,83975,0,Active,<NA>,CMB22-00572,GEOTERRA2023


In [66]:
# Validate every feature geometry and compare structural checks with geometry-derived fields.
geometry_records = []
for feature in raw_data["features"]:
    attributes = feature["attributes"]
    geometry = feature.get("geometry") or {}
    rings = geometry.get("rings") or []
    ring_polygons = [Polygon(ring) for ring in rings if len(ring) >= 4]
    closed_rings = all(ring[0] == ring[-1] for ring in rings if ring)
    valid_rings = all(polygon.is_valid for polygon in ring_polygons)
    positive_ring_area = all(polygon.area > 0 for polygon in ring_polygons)

    geometry_records.append(
        {
            "BLDG_ID": attributes["BLDG_ID"],
            "ring_count": len(rings),
            "rings_closed": closed_rings,
            "rings_valid": valid_rings,
            "positive_ring_area": positive_ring_area,
            "geometry_present": bool(rings),
        }
    )

geometry_detail = pd.DataFrame(geometry_records)
geometry_detail["GEOMETRY_QA_FLAG"] = "OK"
geometry_detail.loc[
    ~geometry_detail["geometry_present"]
    | ~geometry_detail["rings_closed"]
    | ~geometry_detail["rings_valid"]
    | ~geometry_detail["positive_ring_area"],
    "GEOMETRY_QA_FLAG",
] = "Review: geometry structure"
geometry_detail.loc[
    geometry_detail["ring_count"] > 1,
    "GEOMETRY_QA_FLAG",
] = "Review: multipart or interior ring"

analysis_df = analysis_df.merge(
    geometry_detail[["BLDG_ID", "ring_count", "GEOMETRY_QA_FLAG"]],
    on="BLDG_ID",
    how="left",
    validate="one_to_one",
)

geometry_summary = geometry_detail["GEOMETRY_QA_FLAG"].value_counts().rename_axis("flag").to_frame("records")
print("Geometry QA summary")
display(geometry_summary)
print("Geometry-derived zero values")
display(
    df[["Shape.STArea()", "Shape.STLength()"]]
    .eq(0)
    .sum()
    .rename("zero_records")
    .to_frame()
)
print("Invalid or structurally incomplete geometries")
display(
    geometry_detail.loc[
        geometry_detail["GEOMETRY_QA_FLAG"] == "Review: geometry structure"
    ].head(20)
)


Geometry QA summary


,records
flag,
OK,43632
Review: multipart or interior ring,51
Review: geometry structure,3


Geometry-derived zero values


,zero_records
Shape.STArea(),0
Shape.STLength(),0


Invalid or structurally incomplete geometries


,BLDG_ID,ring_count,rings_closed,rings_valid,positive_ring_area,geometry_present,GEOMETRY_QA_FLAG
1204,64559,1,True,False,True,True,Review: geometry structure
17254,81384,1,True,False,True,True,Review: geometry structure
21267,85609,1,True,False,True,True,Review: geometry structure


## Additional Cleaning Decisions

The additional review uses conservative rules:

- Keep all identifier values unchanged. Whitespace-only values would be treated as missing in analysis, but none are rewritten here.
- Keep `ROOF_TYPE` values unchanged unless a verified coding dictionary is available.
- Preserve permit identifiers and expose records with `DEMO_PERMIT` through `DEMO_PERMIT_REVIEW_FLAG`; the current snapshot contains only active status values.
- Keep all geometries and geometry-derived measurements. Multipart or interior-ring features are valid possibilities for building footprints and are flagged for review rather than removed.
- Do not replace values based only on statistical rarity or a mismatch between a source year and construction year.

In [67]:
# Confirm the additional review did not change the source-faithful record set.
invalid_geometry_mask = ~geometry_detail["rings_valid"]
invalid_geometry_count = int(invalid_geometry_mask.sum())

assert len(analysis_df) == len(df)
assert analysis_df["BLDG_ID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert analysis_df["STATUS_LABEL"].notna().all()
assert analysis_df["GEOMETRY_QA_FLAG"].notna().all()
assert geometry_detail["BLDG_ID"].is_unique
assert geometry_detail["geometry_present"].all()
assert geometry_detail["rings_closed"].all()
assert geometry_detail["positive_ring_area"].all()
assert df["Shape.STArea()"].ge(0).all()
assert df["Shape.STLength()"].ge(0).all()
assert analysis_df.loc[invalid_geometry_mask, "GEOMETRY_QA_FLAG"].notna().all()

print("Additional field and geometry validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Multipart or interior-ring features flagged: {(geometry_detail['ring_count'] > 1).sum():,}")
print(f"Invalid geometry features flagged for review: {invalid_geometry_count:,}")
print(f"Records with DEMO_PERMIT review flag: {analysis_df['DEMO_PERMIT_REVIEW_FLAG'].sum():,}")


Additional field and geometry validation passed
Rows preserved: 43,686
Multipart or interior-ring features flagged: 51
Invalid geometry features flagged for review: 4
Records with DEMO_PERMIT review flag: 5


In [68]:
# Diagnose invalid geometries and test reversible repairs without changing raw_data.
from shapely.validation import explain_validity
from shapely import make_valid

geometry_repair_records = []
for feature in raw_data["features"]:
    attributes = feature["attributes"]
    rings = (feature.get("geometry") or {}).get("rings") or []
    source_polygon = Polygon(rings[0]) if len(rings) == 1 and len(rings[0]) >= 4 else None
    source_valid = source_polygon.is_valid if source_polygon is not None else None
    repaired_geometry = (
        make_valid(source_polygon)
        if source_polygon is not None and not source_valid
        else source_polygon
    )

    geometry_repair_records.append(
        {
            "BLDG_ID": attributes["BLDG_ID"],
            "SOURCE_GEOMETRY_VALID": source_valid,
            "SOURCE_GEOMETRY_REASON": (
                explain_validity(source_polygon)
                if source_polygon is not None and not source_valid
                else None
            ),
            "REPAIRED_GEOMETRY_VALID": (
                repaired_geometry.is_valid if repaired_geometry is not None else None
            ),
            "REPAIRED_GEOMETRY_TYPE": (
                repaired_geometry.geom_type if repaired_geometry is not None else None
            ),
            "REPAIR_CHANGED_GEOMETRY": (
                source_polygon is not None
                and repaired_geometry is not None
                and source_polygon.wkb != repaired_geometry.wkb
            ),
        }
    )

geometry_repair_detail = pd.DataFrame(geometry_repair_records)
invalid_geometry_review = geometry_repair_detail.loc[
    geometry_repair_detail["SOURCE_GEOMETRY_VALID"] == False
]

print("Invalid geometry diagnostics")
display(
    invalid_geometry_review[
        [
            "BLDG_ID",
            "SOURCE_GEOMETRY_REASON",
            "REPAIRED_GEOMETRY_VALID",
            "REPAIRED_GEOMETRY_TYPE",
            "REPAIR_CHANGED_GEOMETRY",
        ]
    ]
)
print(f"Single-ring invalid geometries: {len(invalid_geometry_review):,}")
print(
    f"Single-ring invalid geometries repairable by make_valid: "
    f"{invalid_geometry_review['REPAIRED_GEOMETRY_VALID'].sum():,}"
)

geometry_columns = [
    "SOURCE_GEOMETRY_VALID",
    "SOURCE_GEOMETRY_REASON",
    "REPAIRED_GEOMETRY_VALID",
    "REPAIRED_GEOMETRY_TYPE",
    "REPAIR_CHANGED_GEOMETRY",
    "GEOMETRY_REPAIR_RECOMMENDED",
]
analysis_df = analysis_df.drop(columns=geometry_columns, errors="ignore")
analysis_df = analysis_df.merge(
    geometry_repair_detail[
        [
            "BLDG_ID",
            "SOURCE_GEOMETRY_VALID",
            "SOURCE_GEOMETRY_REASON",
            "REPAIRED_GEOMETRY_VALID",
            "REPAIRED_GEOMETRY_TYPE",
            "REPAIR_CHANGED_GEOMETRY",
        ]
    ],
    on="BLDG_ID",
    how="left",
    validate="one_to_one",
)
analysis_df["GEOMETRY_REPAIR_RECOMMENDED"] = (
    analysis_df["SOURCE_GEOMETRY_VALID"].eq(False)
    & analysis_df["REPAIRED_GEOMETRY_VALID"].eq(True)
)


Invalid geometry diagnostics


,BLDG_ID,SOURCE_GEOMETRY_REASON,REPAIRED_GEOMETRY_VALID,REPAIRED_GEOMETRY_TYPE,REPAIR_CHANGED_GEOMETRY
1204,64559,Ring Self-intersection[-13683158.4543963 57064...,True,Polygon,True
17254,81384,Ring Self-intersection[-13689544.9883365 57041...,True,Polygon,True
21267,85609,Ring Self-intersection[-13687301.0044086 57064...,True,Polygon,True


Single-ring invalid geometries: 3
Single-ring invalid geometries repairable by make_valid: 3


In [69]:
# Create a derived geometry view: repair only single-ring self-intersections and retain all other geometry as-is.
analysis_geometry_by_id = {}
for feature in raw_data["features"]:
    attributes = feature["attributes"]
    building_id = attributes["BLDG_ID"]
    rings = (feature.get("geometry") or {}).get("rings") or []

    if len(rings) == 1 and len(rings[0]) >= 4:
        source_polygon = Polygon(rings[0])
        analysis_geometry_by_id[building_id] = (
            make_valid(source_polygon)
            if not source_polygon.is_valid
            else source_polygon
        )
    else:
        analysis_geometry_by_id[building_id] = None

analysis_df["GEOMETRY_ANALYSIS_AVAILABLE"] = analysis_df["BLDG_ID"].map(
    lambda building_id: analysis_geometry_by_id[building_id] is not None
)
analysis_df["GEOMETRY_REPAIRED"] = analysis_df["BLDG_ID"].map(
    lambda building_id: bool(
        geometry_repair_detail.loc[
            geometry_repair_detail["BLDG_ID"] == building_id,
            "REPAIR_CHANGED_GEOMETRY",
        ].iloc[0]
    )
)

print(f"Derived analysis geometries available: {analysis_df['GEOMETRY_ANALYSIS_AVAILABLE'].sum():,}")
print(f"Derived geometries repaired: {analysis_df['GEOMETRY_REPAIRED'].sum():,}")


Derived analysis geometries available: 43,635
Derived geometries repaired: 3


In [70]:
# Check exact duplicate footprints and partial spatial overlaps among single-ring polygons.
from shapely.strtree import STRtree

single_ring_records = []
for feature in raw_data["features"]:
    attributes = feature["attributes"]
    rings = (feature.get("geometry") or {}).get("rings") or []
    if len(rings) == 1 and len(rings[0]) >= 4:
        polygon = analysis_geometry_by_id[attributes["BLDG_ID"]]
        single_ring_records.append(
            {
                "BLDG_ID": attributes["BLDG_ID"],
                "geometry": polygon,
                "geometry_signature": polygon.wkb_hex,
            }
        )

single_ring_geometry = pd.DataFrame(single_ring_records)
exact_duplicate_signatures = (
    single_ring_geometry["geometry_signature"]
    .value_counts()
    .loc[lambda values: values > 1]
)

polygons = single_ring_geometry["geometry"].tolist()
ids = single_ring_geometry["BLDG_ID"].tolist()
tree = STRtree(polygons)
overlap_pairs = set()
for index, polygon in enumerate(polygons):
    for match_index in tree.query(polygon, predicate="overlaps"):
        match_index = int(match_index)
        if index < match_index:
            overlap_pairs.add((ids[index], ids[match_index]))

print(f"Single-ring geometries checked: {len(single_ring_geometry):,}")
print(f"Exact duplicate footprint signatures: {len(exact_duplicate_signatures):,}")
print(f"Partial-overlap footprint pairs: {len(overlap_pairs):,}")

if not exact_duplicate_signatures.empty:
    display(exact_duplicate_signatures.rename("records").to_frame())


Single-ring geometries checked: 43,635
Exact duplicate footprint signatures: 0
Partial-overlap footprint pairs: 20


In [71]:
# Inspect source field metadata and define a compact analytical schema.
schema_records = []
for field in raw_data["fields"]:
    schema_records.append(
        {
            "field": field.get("name"),
            "type": field.get("type"),
            "alias": field.get("alias"),
            "length": field.get("length"),
            "has_domain": field.get("domain") is not None,
        }
    )

schema_df = pd.DataFrame(schema_records)
display(schema_df)

completely_missing_fields = [
    field
    for field, classification_name in field_classification.items()
    if classification_name == "Completely Missing"
]
analysis_columns = [
    column for column in analysis_df.columns
    if column not in completely_missing_fields
]
analysis_df_compact = analysis_df[analysis_columns].copy()

cleaning_manifest = pd.DataFrame(
    [
        {
            "field_or_scope": "YEAR_BUILT",
            "action": "Replace zero sentinel with missing in analysis_df",
            "source_preserved": True,
            "review_flag": "YEAR_BUILT_QA_FLAG",
        },
        {
            "field_or_scope": "HEIGHT",
            "action": "Replace zero sentinel with missing in analysis_df",
            "source_preserved": True,
            "review_flag": "HEIGHT_ZERO_FLAG",
        },
        {
            "field_or_scope": "Whitespace-only identifiers and ROOF_TYPE",
            "action": "Represent as missing in derived clean columns",
            "source_preserved": True,
            "review_flag": "None",
        },
        {
            "field_or_scope": "Single-ring invalid geometry",
            "action": "Use make_valid in derived geometry view",
            "source_preserved": True,
            "review_flag": "GEOMETRY_REPAIR_RECOMMENDED",
        },
        {
            "field_or_scope": "Multipart/interior-ring and overlap features",
            "action": "Retain and flag for contextual review",
            "source_preserved": True,
            "review_flag": "GEOMETRY_QA_FLAG",
        },
        {
            "field_or_scope": "Completely empty source fields",
            "action": "Exclude from analysis_df_compact only",
            "source_preserved": True,
            "review_flag": "None",
        },
    ]
)

print(f"Source fields: {len(schema_df):,}")
print(f"Compact analytical fields: {len(analysis_df_compact.columns):,}")
print("Cleaning manifest")
display(cleaning_manifest)


,field,type,alias,length,has_domain
0,OBJECTID,esriFieldTypeOID,OBJECTID,NaN,False
1,BLDG_ID,esriFieldTypeInteger,Building ID,NaN,False
2,STATUS,esriFieldTypeString,Status,1.0,False
3,NUM_STORIES,esriFieldTypeSmallInteger,Number of Stories,NaN,False
4,HEIGHT,esriFieldTypeDouble,Height in Feet,NaN,False
5,YEAR_BUILT,esriFieldTypeSmallInteger,Year Built,NaN,False
6,SOURCE,esriFieldTypeString,Data Source,16.0,False
7,PLANREFID,esriFieldTypeString,Fire Preplan ID,10.0,False
8,CONST_TYPE,esriFieldTypeString,Construction Type,50.0,False
9,ROOF_TYPE,esriFieldTypeString,Roof Type,50.0,False


Source fields: 26
Compact analytical fields: 46
Cleaning manifest


,field_or_scope,action,source_preserved,review_flag
0,YEAR_BUILT,Replace zero sentinel with missing in analysis_df,True,YEAR_BUILT_QA_FLAG
1,HEIGHT,Replace zero sentinel with missing in analysis_df,True,HEIGHT_ZERO_FLAG
2,Whitespace-only identifiers and ROOF_TYPE,Represent as missing in derived clean columns,True,None
3,Single-ring invalid geometry,Use make_valid in derived geometry view,True,GEOMETRY_REPAIR_RECOMMENDED
4,Multipart/interior-ring and overlap features,Retain and flag for contextual review,True,GEOMETRY_QA_FLAG
5,Completely empty source fields,Exclude from analysis_df_compact only,True,None


## Resolution of Unresolved Cleaning Items

The unresolved items are resolved conservatively for analysis:

- Invalid single-ring self-intersections are repaired with `make_valid` only in `analysis_geometry_by_id`; the raw ArcGIS rings remain unchanged.
- Multipart or interior-ring features are retained and flagged because those structures can represent legitimate building footprints.
- Exact duplicate footprints are not present. Partial overlaps are retained for contextual review because overlap alone does not prove duplication.
- Identifier repetition is retained because `PLANREFID`, `PERMIT_ID`, `DEMO_PERMIT`, and `OMS_FACILITY_ID` describe relationships and facilities rather than unique building records.
- Completely empty fields are excluded only from `analysis_df_compact`; they remain in `df` and the raw dataset.
- The ArcGIS schema confirms that `HEIGHT` is measured in feet. Area and length units are retained as supplied by the source service.

In [72]:
# Final checks for the derived processed-analysis views.
empty_source_fields_in_compact = [
    field for field in completely_missing_fields
    if field in analysis_df_compact.columns
]

assert len(raw_data["features"]) == len(df) == len(analysis_df)
assert analysis_df["BLDG_ID"].is_unique
assert set(analysis_geometry_by_id) == set(analysis_df["BLDG_ID"])
assert invalid_geometry_review["REPAIRED_GEOMETRY_VALID"].eq(True).all()
assert invalid_geometry_review["REPAIR_CHANGED_GEOMETRY"].eq(True).all()
assert len(exact_duplicate_signatures) == 0
assert not empty_source_fields_in_compact
assert analysis_df["HEIGHT"].dropna().ge(0).all()
assert analysis_df["Shape.STArea()"].ge(0).all()
assert analysis_df["Shape.STLength()"].ge(0).all()

print("Processed-analysis cleaning validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Compact analytical fields: {len(analysis_df_compact.columns):,}")
print(f"Repaired single-ring geometries: {len(invalid_geometry_review):,}")
print(f"Partial-overlap pairs retained for review: {len(overlap_pairs):,}")


Processed-analysis cleaning validation passed
Rows preserved: 43,686
Compact analytical fields: 46
Repaired single-ring geometries: 3
Partial-overlap pairs retained for review: 20


## Export Processed Dataset

The processed dataset will be written separately from the raw download. It will contain the compact analytical attributes, derived QA fields, and ArcGIS-compatible geometry. Repaired geometry is used only for the three single-ring self-intersections; multipart or interior-ring geometry is retained from the raw feature.

In [73]:
# Serialize the processed analytical attributes and geometry to separate JSON files.
from datetime import datetime
import math
import numpy as np


def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def geometry_to_arcgis_rings(geometry):
    if geometry is None:
        return None

    if geometry.geom_type == "Polygon":
        polygons = [geometry]
    elif geometry.geom_type == "MultiPolygon":
        polygons = list(geometry.geoms)
    else:
        return None

    rings = []
    for polygon in polygons:
        rings.append([[json_safe(x), json_safe(y)] for x, y in polygon.exterior.coords])
        rings.extend(
            [
                [[json_safe(x), json_safe(y)] for x, y in interior.coords]
                for interior in polygon.interiors
            ]
        )
    return {"rings": rings}


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_FILE = PROCESSED_DIR / "HIL-005_cleaned.json"
MANIFEST_FILE = PROCESSED_DIR / "HIL-005_cleaning_manifest.json"

processed_features = []
for raw_feature in raw_data["features"]:
    building_id = raw_feature["attributes"]["BLDG_ID"]
    row = analysis_df_compact.loc[
        analysis_df_compact["BLDG_ID"] == building_id
    ].iloc[0]
    attributes = {
        field: json_safe(value)
        for field, value in row.to_dict().items()
    }

    derived_geometry = analysis_geometry_by_id[building_id]
    geometry = (
        geometry_to_arcgis_rings(derived_geometry)
        if derived_geometry is not None
        else raw_feature.get("geometry")
    )
    processed_features.append(
        {
            "attributes": attributes,
            "geometry": geometry,
        }
    )

processed_data = {
    "displayFieldName": raw_data.get("displayFieldName"),
    "geometryType": raw_data.get("geometryType"),
    "spatialReference": raw_data.get("spatialReference"),
    "fields": schema_records,
    "features": processed_features,
    "cleaning_summary": {
        "source_file": RAW_FILE.name,
        "data_version": DATA_VERSION,
        "rows": len(processed_features),
        "compact_attribute_fields": len(analysis_df_compact.columns),
        "repaired_single_ring_geometries": int(analysis_df["GEOMETRY_REPAIRED"].sum()),
        "multipart_or_interior_ring_features": int((analysis_df["ring_count"] > 1).sum()),
        "partial_overlap_pairs_retained_for_review": len(overlap_pairs),
    },
}

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, indent=2, ensure_ascii=True)

manifest_data = {
    "dataset": "HIL-005",
    "data_version": DATA_VERSION,
    "created_utc": datetime.now().astimezone().isoformat(),
    "source_file": str(RAW_FILE),
    "processed_file": str(PROCESSED_FILE),
    "rows": len(processed_features),
    "cleaning_manifest": [
        {
            key: json_safe(value)
            for key, value in record.items()
        }
        for record in cleaning_manifest.to_dict(orient="records")
    ],
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as file:
    json.dump(manifest_data, file, indent=2, ensure_ascii=True)

with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
    reloaded_processed_data = json.load(file)
with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    reloaded_manifest_data = json.load(file)

assert len(reloaded_processed_data["features"]) == len(raw_data["features"])
assert len(reloaded_manifest_data["cleaning_manifest"]) == len(cleaning_manifest)
assert reloaded_processed_data["features"][0]["attributes"]["BLDG_ID"] == int(df.iloc[0]["BLDG_ID"])

print(f"Wrote: {PROCESSED_FILE}")
print(f"Wrote: {MANIFEST_FILE}")
print(f"Processed features: {len(reloaded_processed_data['features']):,}")
print("Reload validation passed")


Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-005_cleaned.json
Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-005_cleaning_manifest.json
Processed features: 43,686
Reload validation passed
